# Zarr Store Inspector

Point this notebook at any local or GCS Zarr store to get a complete picture of its contents before writing any retrieval code.

In [ ]:
from pathlib import Path
import sys

import gcsfs
import numpy as np
import xarray as xr
import pandas as pd

PROJECT_ROOT = Path.cwd().parent if Path.cwd().name == "notebooks" else Path.cwd()
SRC_PATH = PROJECT_ROOT / "src"
if str(SRC_PATH) not in sys.path:
    sys.path.insert(0, str(SRC_PATH))

print("Project root:", PROJECT_ROOT)

## Configure Store Path

Set `STORE_PATH` to a local directory or a `gs://` URI.  
Set `USE_ANONYMOUS` to `False` if the bucket requires credentials.

In [ ]:
#STORE_PATH = "gs://gcp-public-data-arco-era5/ar/full_37-1h-0p25deg-chunk-1.zarr-v3"
STORE_PATH = (PROJECT_ROOT / "data" / "sample_met.zarr").as_posix()
USE_ANONYMOUS = True   # set False to use application-default credentials

def open_store(path: str, anonymous: bool = True) -> xr.Dataset:
    if path.startswith("gs://"):
        token = "anon" if anonymous else "google_default"
        fs = gcsfs.GCSFileSystem(token=token)
        mapper = fs.get_mapper(path)
        return xr.open_zarr(mapper, consolidated=True, chunks={})
    return xr.open_zarr(path, consolidated=True, chunks={})

ds = open_store(STORE_PATH, anonymous=USE_ANONYMOUS)
print("Store opened:", STORE_PATH)

## Dimensions

In [ ]:
dim_rows = [{"dimension": k, "size": v} for k, v in ds.sizes.items()]
pd.DataFrame(dim_rows).set_index("dimension")

## Coordinates

Shape, dtype, units/calendar where present, and first/last values.

In [ ]:
coord_rows = []
for name, coord in ds.coords.items():
    attrs = coord.attrs
    vals = coord.values
    first = vals.flat[0] if vals.size > 0 else None
    last  = vals.flat[-1] if vals.size > 0 else None
    coord_rows.append({
        "coordinate": name,
        "dims": str(coord.dims),
        "shape": str(coord.shape),
        "dtype": str(coord.dtype),
        "units / calendar": attrs.get("units", attrs.get("calendar", "")),
        "first": first,
        "last": last,
    })
pd.DataFrame(coord_rows).set_index("coordinate")

## Data Variables

Shape, dtype, units, long_name, and estimated uncompressed size per variable.

In [ ]:
var_rows = []
for name, da in ds.data_vars.items():
    attrs = da.attrs
    nbytes = int(np.prod(da.shape)) * np.dtype(da.dtype).itemsize
    var_rows.append({
        "variable": name,
        "dims": str(da.dims),
        "shape": str(da.shape),
        "dtype": str(da.dtype),
        "units": attrs.get("units", ""),
        "long_name": attrs.get("long_name", ""),
        "uncompressed MiB": round(nbytes / 1024**2, 1),
    })
pd.DataFrame(var_rows).set_index("variable")

## Chunk Layout

Understanding chunks is critical for efficient bounding-box retrieval — ideally your spatial slice aligns with chunk boundaries.

In [ ]:
chunk_rows = []
for name in ds.data_vars:
    encoding = ds[name].encoding
    chunk_rows.append({
        "variable": name,
        "chunks": encoding.get("chunks", "not set"),
        "compressor": str(encoding.get("compressor", "none")),
        "dtype (stored)": encoding.get("dtype", ds[name].dtype),
    })
pd.DataFrame(chunk_rows).set_index("variable")

## Dataset-level Attributes

In [ ]:
if ds.attrs:
    for k, v in ds.attrs.items():
        print(f"{k}: {v}")
else:
    print("No dataset-level attributes.")

## Time Coverage

In [ ]:
time_candidates = [c for c in ds.coords if "time" in c.lower()]
for tc in time_candidates:
    t = ds[tc]
    vals = t.values
    n = t.size
    step = (vals[1] - vals[0]) if n > 1 else "N/A"
    print(f"{tc}: {vals[0]}  →  {vals[-1]}  ({n} steps, step={step})")

## Vertical Coordinate

In [ ]:
level_candidates = [c for c in ds.coords if any(k in c.lower() for k in ["level", "pressure", "isobaric", "height"])]
for lc in level_candidates:
    lv = ds[lc].values
    print(f"{lc}: {lv}")

## Spatial Extent &amp; Resolution

In [ ]:
spatial_keywords = {"latitude", "longitude", "lat", "lon"}
for coord_name in ds.coords:
    if coord_name.lower() in spatial_keywords or any(coord_name.lower().startswith(k) for k in spatial_keywords):
        vals = ds[coord_name].values
        step = round(float(np.diff(vals[:2])[0]), 6) if len(vals) > 1 else "N/A"
        print(f"{coord_name}: min={vals.min():.4f}  max={vals.max():.4f}  step={step}  n={len(vals)}")

In [ ]:
var_rows


In [ ]:
# CAREFUL: Optional plotting code
import matplotlib.pyplot as plt

# Candidate variables to visualize if present in this store.
plot_candidates = [
    "temperature",
    "u_component_of_wind",
    "v_component_of_wind",
    "surface_pressure",
    "boundary_layer_height",
]

# Resolve coordinate names defensively.
time_dim = next((d for d in ds.dims if "time" in d.lower()), None)
lat_dim = next((d for d in ds.dims if "lat" in d.lower()), None)
lon_dim = next((d for d in ds.dims if "lon" in d.lower()), None)
level_dim = next((d for d in ds.dims if any(k in d.lower() for k in ["level", "pressure", "isobaric", "height"])), None)

selected = [v for v in plot_candidates if v in ds.data_vars][:3]
if not selected:
    raise ValueError(f"None of the candidate variables were found: {plot_candidates}")

ncols = len(selected)
fig, axes = plt.subplots(1, ncols, figsize=(6 * ncols, 5), constrained_layout=True)
if ncols == 1:
    axes = [axes]

for ax, var_name in zip(axes, selected):
    da = ds[var_name]

    # Choose a representative slice that yields a 2D lat/lon field when possible.
    indexers = {}
    if time_dim and time_dim in da.dims:
        indexers[time_dim] = 0
    if level_dim and level_dim in da.dims:
        indexers[level_dim] = 0

    field = da.isel(indexers) if indexers else da

    # If still >2D, drop extra dims by taking the first index.
    extra_dims = [d for d in field.dims if d not in {lat_dim, lon_dim}]
    for d in extra_dims:
        field = field.isel({d: 0})

    if lat_dim in field.dims and lon_dim in field.dims:
        field.plot(ax=ax, robust=True)
        ax.set_title(f"{var_name} (sample slice)")
    else:
        # Fallback: simple line plot over first available dimension.
        field.plot(ax=ax)
        ax.set_title(f"{var_name} (1D fallback)")

plt.show()

In [ ]:
ds.close()